# SPARC Example 13: The Mass Discrepancy at R_max

**EPS Research RAG Astrophysics Corpus — Unified HI Corpus v7.0**

The ratio Vobs/Vbar at the outermost ring quantifies the mass discrepancy —
how much more mass is implied by kinematics than accounted for by baryons.
This is the central observational fact that dark matter, MOND, and the
omega correction each try to explain.

**Important note on corpus fidelity:** The `rotation_curve_corpus_v7_flat.csv` and `rotation_curve_corpus_v7.json` are **full-fidelity** — not a summary or veneer. The CSV contains every kinematic parameter published by Lelli et al. (2016) including per-galaxy inclination, distance uncertainties, mass-to-light ratios, and rotation curve statistics. The JSON adds full per-ring data: Vobs, Vgas, Vdisk, Vbul, errV at every radial point. This is the complete published dataset in a single machine-readable file.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.19563417  
**Source:** Lelli, McGaugh & Schombert (2016), AJ 152, 157  
**Dependencies:** Python 3, numpy, matplotlib, csv (standard library only)

In [ ]:
# ── Colab setup: download corpus from Zenodo ──────────────────
import os, urllib.request
CORPORA = {
    'rotation_curve_corpus_v7.json': 'https://zenodo.org/records/19563417/files/rotation_curve_corpus_v7.json',
}
for filename, url in CORPORA.items():
    if not os.path.exists(filename):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, filename)
        print(f"  ✓ {filename}")
    else:
        print(f"  Already present: {filename}")
print("Ready.")


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open('rotation_curve_corpus_v7.json') as f:
    corpus = json.load(f)

results = []
for g in corpus['galaxies']:
    if g['survey'] != 'SPARC' or not g.get('data'):
        continue
    last = g['data'][-1]
    Vobs = last.get('Vobs', 0)
    Vgas = last.get('Vgas', 0)
    Vdisk= last.get('Vdisk', 0)
    Vbul = last.get('Vbul', 0)
    Vbar_sq = Vgas**2 + Vdisk**2 + Vbul**2
    if Vbar_sq <= 0 or Vobs <= 0:
        continue
    Vbar = np.sqrt(Vbar_sq)
    results.append({'galaxy': g['galaxy'], 'Vobs': Vobs,
                    'Vbar': Vbar, 'ratio': Vobs/Vbar})

ratios = [r['ratio'] for r in results]
print(f"SPARC galaxies with baryonic data: {len(results)}")
print(f"Median Vobs/Vbar at R_max: {np.median(ratios):.2f}")
print(f"Max ratio: {max(ratios):.2f} ({results[ratios.index(max(ratios))]['galaxy']})")
print(f"Min ratio: {min(ratios):.2f} ({results[ratios.index(min(ratios))]['galaxy']})")
print(f"\nRatio > 2 (strong dark matter excess): "
      f"{sum(1 for r in ratios if r > 2)} galaxies")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
vbars  = [r['Vbar']  for r in results]
ratios = [r['ratio'] for r in results]
sc = ax.scatter(vbars, ratios, s=20, alpha=0.6, c=ratios,
                cmap='RdYlBu_r', vmin=0.5, vmax=3.5)
plt.colorbar(sc, ax=ax, label=r'$V_{\rm obs}/V_{\rm bar}$')
ax.axhline(1.0, color='black', linestyle='--', linewidth=1, alpha=0.5,
           label=r'$V_{\rm obs} = V_{\rm bar}$ (no discrepancy)')
ax.set_xlabel(r'$V_{\rm bar}$ at $R_{\rm max}$ (km/s)', fontsize=12)
ax.set_ylabel(r'$V_{\rm obs}/V_{\rm bar}$', fontsize=12)
ax.set_title('Mass Discrepancy at Outermost Ring — SPARC\n'
             'EPS Research Corpus v7.0', fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('ex13_mass_discrepancy.png', dpi=150, bbox_inches='tight')
plt.show()